# Retries, Fallbacks, and Circuit Breakers
Companion notebook for the **Retries, Fallbacks, and Circuit Breakers** section of Chapter 8.

Retry only when the same request has a real chance to work (timeouts, rate limits, short-lived 5xx). Don't retry invalid input, auth failures, or permission errors -- repeating the same call will just fail again.

This is the book's `call_with_retry`. One fix from the printed text: the retry-decision logic (`status`, `retryable`, the backoff sleep) has to live *inside* the `except` block -- it reads `e` and needs to run only after a failure, not after every attempt.

In [1]:
import random
import time

RETRYABLE_STATUS = {429, 500, 503}


def call_with_retry(tool_call, max_attempts=3, base_delay=0.5):
    for attempt in range(1, max_attempts + 1):
        try:
            return tool_call()
        except Exception as e:
            status = getattr(e, "status_code", None)
            retryable = status in RETRYABLE_STATUS or isinstance(e, TimeoutError)
            if not retryable or attempt == max_attempts:
                raise
            delay = base_delay * (2 ** (attempt - 1))
            jitter = random.uniform(0, delay * 0.25)
            time.sleep(delay + jitter)

A flaky call that fails with a retryable `503` twice, then succeeds -- and a bad-input call that must *not* be retried:

In [2]:
class FakeAPIError(Exception):
    def __init__(self, status_code):
        super().__init__(f"status {status_code}")
        self.status_code = status_code


def make_flaky_call(fail_times: int, status_code: int):
    state = {"calls": 0}

    def call():
        state["calls"] += 1
        if state["calls"] <= fail_times:
            raise FakeAPIError(status_code)
        return {"ok": True, "attempts_needed": state["calls"]}

    return call, state


flaky_call, flaky_state = make_flaky_call(fail_times=2, status_code=503)
result = call_with_retry(flaky_call, max_attempts=3, base_delay=0.01)
print("retryable failure recovered:", result, "| total calls:", flaky_state["calls"])

bad_call, bad_state = make_flaky_call(fail_times=99, status_code=401)
try:
    call_with_retry(bad_call, max_attempts=3, base_delay=0.01)
except FakeAPIError as e:
    print(f"non-retryable {e.status_code} raised immediately after {bad_state['calls']} call(s)")

retryable failure recovered: {'ok': True, 'attempts_needed': 3} | total calls: 3
non-retryable 401 raised immediately after 1 call(s)


**Fallback** is different from retry: instead of trying the same path again, switch to another one -- a smaller model, a cached answer, read-only mode, or human review.

In [3]:
def answer_with_fallback(tool_call, fallback_response):
    try:
        return call_with_retry(tool_call, max_attempts=2, base_delay=0.01)
    except Exception:
        return {"degraded": True, "answer": fallback_response}


always_fails, _ = make_flaky_call(fail_times=99, status_code=503)
print(answer_with_fallback(always_fails, fallback_response="cached summary from last known-good run"))

{'degraded': True, 'answer': 'cached summary from last known-good run'}


A **circuit breaker** stops calling a dependency that's already failing, instead of letting retries pile up: **closed** (normal), **open** (fail fast, no calls), **half-open** (let a few probes through to check recovery).

In [4]:
import time as _time


class CircuitBreaker:
    def __init__(self, failure_threshold=3, reset_after_s=5):
        self.failure_threshold = failure_threshold
        self.reset_after_s = reset_after_s
        self.state = "closed"
        self.failure_count = 0
        self.opened_at = None

    def call(self, tool_call):
        if self.state == "open":
            if _time.monotonic() - self.opened_at < self.reset_after_s:
                raise RuntimeError("circuit open: failing fast, dependency looks unhealthy")
            self.state = "half_open"  # wait period elapsed; let one probe through

        try:
            result = tool_call()
        except Exception:
            self.failure_count += 1
            if self.state == "half_open" or self.failure_count >= self.failure_threshold:
                self.state = "open"
                self.opened_at = _time.monotonic()
            raise

        self.state = "closed"
        self.failure_count = 0
        return result

In [5]:
breaker = CircuitBreaker(failure_threshold=3, reset_after_s=0.05)
always_fails, _ = make_flaky_call(fail_times=99, status_code=500)

for i in range(3):
    try:
        breaker.call(always_fails)
    except Exception as e:
        print(f"attempt {i+1}: {type(e).__name__} -- breaker state = {breaker.state}")

try:
    breaker.call(always_fails)
except RuntimeError as e:
    print(f"attempt 4 (breaker open, fails fast without calling the dependency): {e}")

import time
time.sleep(0.06)  # let reset_after_s elapse

recovering_call, _ = make_flaky_call(fail_times=0, status_code=500)
print("attempt 5 (after wait, half-open probe):", breaker.call(recovering_call), "-> breaker state =", breaker.state)

attempt 1: FakeAPIError -- breaker state = closed
attempt 2: FakeAPIError -- breaker state = closed
attempt 3: FakeAPIError -- breaker state = open
attempt 4 (breaker open, fails fast without calling the dependency): circuit open: failing fast, dependency looks unhealthy
attempt 5 (after wait, half-open probe): {'ok': True, 'attempts_needed': 1} -> breaker state = closed
